In [ ]:
# ============================================================
# FINAL DATASET CLEANING + PREPROCESSING SCRIPT
# Train / Val / Test split with:
# - corrupted image removal
# - exact duplicate removal
# - N4 bias correction
# - training-only vertical flip
# - training-only Gaussian noise
# - training-only elastic distortion
# - manifest/log CSV files for reproducibility
# ============================================================

# In Colab, run once:
# !pip install SimpleITK tqdm opencv-python

import os
import cv2
import csv
import shutil
import random
import hashlib
import numpy as np
from tqdm import tqdm
from collections import Counter, defaultdict

try:
    import SimpleITK as sitk
    SIMPLEITK_AVAILABLE = True
except Exception:
    SIMPLEITK_AVAILABLE = False


# ============================================================
# CONFIGURATION
# ============================================================

raw_path = "raw dataset here"
clean_path = "Output"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TRAIN_RATIO = 0.70
VAL_RATIO = 0.10
TEST_RATIO = 0.20

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-8

TARGET_SIZE = (380, 380)
RESET_OUTPUT_FOLDER = True

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Training-only augmentation probabilities
P_VERTICAL_FLIP = 0.30
P_GAUSSIAN_NOISE = 0.30
P_ELASTIC_DISTORTION = 0.20


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def is_image_file(filename):
    return os.path.splitext(filename.lower())[1] in IMG_EXTS


def md5_hash(path, chunk_size=8192):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def save_csv(rows, output_file, fieldnames):
    with open(output_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def resize_with_padding(img, target_size=(224, 224)):
    target_w, target_h = target_size
    h, w = img.shape[:2]

    scale = min(target_w / w, target_h / h)
    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    pad_w = target_w - new_w
    pad_h = target_h - new_h

    left = pad_w // 2
    right = pad_w - left
    top = pad_h // 2
    bottom = pad_h - top

    padded = cv2.copyMakeBorder(
        resized,
        top,
        bottom,
        left,
        right,
        borderType=cv2.BORDER_CONSTANT,
        value=[0, 0, 0]
    )

    return padded


# ============================================================
# N4 BIAS CORRECTION
# ============================================================

def apply_n4_bias_correction(img):
    """
    Applies N4 bias field correction.
    If SimpleITK is unavailable, returns the original image.
    """

    if not SIMPLEITK_AVAILABLE:
        return img

    try:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray_float = gray.astype(np.float32)

        sitk_img = sitk.GetImageFromArray(gray_float)
        mask = sitk.OtsuThreshold(sitk_img, 0, 1, 200)

        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        corrected = corrector.Execute(sitk_img, mask)

        corrected_np = sitk.GetArrayFromImage(corrected)
        corrected_np = cv2.normalize(corrected_np, None, 0, 255, cv2.NORM_MINMAX)
        corrected_np = corrected_np.astype(np.uint8)

        return cv2.cvtColor(corrected_np, cv2.COLOR_GRAY2BGR)

    except Exception:
        return img


# ============================================================
# TRAINING-ONLY AUGMENTATION
# ============================================================

def add_gaussian_noise(img, mean=0, sigma=8):
    noise = np.random.normal(mean, sigma, img.shape).astype(np.float32)
    noisy = img.astype(np.float32) + noise
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)
    return noisy


def elastic_distortion(img, alpha=8, sigma=4):
    h, w = img.shape[:2]

    dx = cv2.GaussianBlur((np.random.rand(h, w) * 2 - 1), (0, 0), sigma) * alpha
    dy = cv2.GaussianBlur((np.random.rand(h, w) * 2 - 1), (0, 0), sigma) * alpha

    x, y = np.meshgrid(np.arange(w), np.arange(h))

    map_x = (x + dx).astype(np.float32)
    map_y = (y + dy).astype(np.float32)

    distorted = cv2.remap(
        img,
        map_x,
        map_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101
    )

    return distorted


def apply_training_augmentation(img):
    if random.random() < P_VERTICAL_FLIP:
        img = cv2.flip(img, 0)

    if random.random() < P_GAUSSIAN_NOISE:
        img = add_gaussian_noise(img, sigma=8)

    if random.random() < P_ELASTIC_DISTORTION:
        img = elastic_distortion(img, alpha=8, sigma=4)

    return img


# ============================================================
# STEP 1: COLLECT CLEAN UNIQUE IMAGES
# ============================================================

def collect_clean_unique_images(folder_path):
    records = []
    corrupted_records = []
    duplicate_records = []

    seen_hashes = {}

    class_names = sorted([
        d for d in os.listdir(folder_path)
        if os.path.isdir(os.path.join(folder_path, d))
    ])

    print("Detected classes:", class_names)

    for cls in class_names:
        cls_path = os.path.join(folder_path, cls)

        files = sorted([
            f for f in os.listdir(cls_path)
            if os.path.isfile(os.path.join(cls_path, f))
        ])

        for fname in tqdm(files, desc=f"Scanning {cls}"):
            if not is_image_file(fname):
                continue

            path = os.path.join(cls_path, fname)

            img = cv2.imread(path)
            if img is None:
                corrupted_records.append({
                    "path": path,
                    "class": cls,
                    "reason": "Unreadable by OpenCV"
                })
                continue

            try:
                file_hash = md5_hash(path)
            except Exception as e:
                corrupted_records.append({
                    "path": path,
                    "class": cls,
                    "reason": f"Hash error: {str(e)}"
                })
                continue

            if file_hash in seen_hashes:
                duplicate_records.append({
                    "removed_path": path,
                    "removed_class": cls,
                    "kept_path": seen_hashes[file_hash]["path"],
                    "kept_class": seen_hashes[file_hash]["class"],
                    "md5": file_hash
                })
                continue

            seen_hashes[file_hash] = {
                "path": path,
                "class": cls
            }

            records.append({
                "source_path": path,
                "filename": fname,
                "class": cls,
                "md5": file_hash
            })

    return records, corrupted_records, duplicate_records, class_names


# ============================================================
# STEP 2: STRATIFIED TRAIN / VAL / TEST SPLIT
# ============================================================

def stratified_split(records):
    rng = random.Random(SEED)

    by_class = defaultdict(list)

    for r in records:
        by_class[r["class"]].append(r)

    split_records = []

    for cls in sorted(by_class.keys()):
        items = sorted(by_class[cls], key=lambda x: x["source_path"])
        rng.shuffle(items)

        n = len(items)

        n_train = int(n * TRAIN_RATIO)
        n_val = int(n * VAL_RATIO)

        train_items = items[:n_train]
        val_items = items[n_train:n_train + n_val]
        test_items = items[n_train + n_val:]

        for split_name, split_items in [
            ("train", train_items),
            ("val", val_items),
            ("test", test_items)
        ]:
            for item in split_items:
                new_item = item.copy()
                new_item["split"] = split_name
                split_records.append(new_item)

    return split_records


# ============================================================
# STEP 3: PREPROCESS AND SAVE IMAGES
# ============================================================

def preprocess_and_save(split_records, output_path):
    if RESET_OUTPUT_FOLDER and os.path.exists(output_path):
        shutil.rmtree(output_path)

    os.makedirs(output_path, exist_ok=True)

    for r in tqdm(split_records, desc="Preprocessing and saving"):
        split_name = r["split"]
        cls = r["class"]
        source_path = r["source_path"]

        img = cv2.imread(source_path)
        if img is None:
            continue

        # Apply N4 bias correction to all splits
        img = apply_n4_bias_correction(img)

        # Resize to 224x224 while preserving aspect ratio
        img = resize_with_padding(img, TARGET_SIZE)

        # Apply augmentation only to training images
        if split_name == "train":
            img = apply_training_augmentation(img)

        dst_dir = os.path.join(output_path, split_name, cls)
        os.makedirs(dst_dir, exist_ok=True)

        ext = os.path.splitext(r["filename"])[1].lower()
        base_name = os.path.splitext(r["filename"])[0]
        safe_filename = f"{r['md5'][:12]}_{base_name}{ext}"

        dst_path = os.path.join(dst_dir, safe_filename)

        cv2.imwrite(dst_path, img)

        r["destination_path"] = dst_path
        r["saved_filename"] = safe_filename

    return split_records


# ============================================================
# STEP 4: SAVE LOGS AND MANIFEST
# ============================================================

def save_outputs(split_records, corrupted_records, duplicate_records, class_names, output_path):
    manifest_fields = [
        "source_path",
        "destination_path",
        "filename",
        "saved_filename",
        "class",
        "split",
        "md5"
    ]

    save_csv(
        split_records,
        os.path.join(output_path, "split_manifest.csv"),
        manifest_fields
    )

    save_csv(
        corrupted_records,
        os.path.join(output_path, "removed_corrupted.csv"),
        ["path", "class", "reason"]
    )

    save_csv(
        duplicate_records,
        os.path.join(output_path, "removed_exact_duplicates.csv"),
        ["removed_path", "removed_class", "kept_path", "kept_class", "md5"]
    )

    with open(os.path.join(output_path, "class_names.txt"), "w") as f:
        for cls in class_names:
            f.write(cls + "\n")


# ============================================================
# STEP 5: SHOW SUMMARY
# ============================================================

def show_summary(split_records):
    print("\nDataset Summary")

    counts = Counter((r["split"], r["class"]) for r in split_records)

    all_splits = ["train", "val", "test"]
    all_classes = sorted(set(r["class"] for r in split_records))

    grand_total = 0

    for split_name in all_splits:
        print(f"\n{split_name.upper()} DATA:")

        split_total = 0

        for cls in all_classes:
            n = counts[(split_name, cls)]
            split_total += n
            print(f"  {cls}: {n} images")

        grand_total += split_total
        print(f"  Total {split_name}: {split_total}")

    print(f"\nGrand total used images: {grand_total}")


# ============================================================
# RUN SCRIPT
# ============================================================

print("Starting final dataset cleaning and preprocessing...")

records, corrupted_records, duplicate_records, class_names = collect_clean_unique_images(raw_path)

print(f"\nReadable unique images: {len(records)}")
print(f"Corrupted/unreadable images removed: {len(corrupted_records)}")
print(f"Exact duplicate images removed: {len(duplicate_records)}")

split_records = stratified_split(records)

split_records = preprocess_and_save(split_records, clean_path)

save_outputs(
    split_records=split_records,
    corrupted_records=corrupted_records,
    duplicate_records=duplicate_records,
    class_names=class_names,
    output_path=clean_path
)

show_summary(split_records)

print("\nDataset cleaning and preprocessing completed successfully.")
print(f"Output folder: {clean_path}")
print("Generated files:")
print("  split_manifest.csv")
print("  removed_corrupted.csv")
print("  removed_exact_duplicates.csv")
print("  class_names.txt")